# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library, following the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"
# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Display dataset metadata fields
print("Dataset Title:", dataset.metadata.name)
print("Description:")
print(dataset.metadata.description)
print("\nPublished on:", dataset.metadata.datePublished)
print("\nLicense:", dataset.metadata.license)
print("\nCite as:", getattr(dataset.metadata, 'citeAs', '(cite as unavailable)'))

## 2. Data Overview
Review available record sets, fields, and their IDs.

Here we inspect the main record sets, their `@id`s, and contained fields (columns).

In [ ]:
# List all record sets in the dataset and their fields/columns by @id
record_set_ids = []
print("Available record sets and their fields (referenced by @id):\n")
for recset in dataset.metadata.recordSet:
    recset_id = recset['@id'] if isinstance(recset, dict) and '@id' in recset else recset
    record_set_ids.append(recset_id)
    print(f"- Record Set @id: {recset_id}")
    # Show the name if present
    if isinstance(recset, dict):
        print(f"  Name: {recset.get('name', '(no name)')}")
        # List columns or fields if present
        if 'field' in recset:
            print("  Fields (by @id):")
            for fld in recset['field']:
                field_id = fld['@id'] if isinstance(fld, dict) and '@id' in fld else fld
                print(f"    - {field_id}")
    print("")

if not record_set_ids:
    print("No record sets found in the top-level metadata.\nAttempting to infer record set @id(s) from the distribution block...")
    # Try to probe via dataset.metadata.distribution (often refers to the file, which is referenced as a recordSet)
    for obj in dataset.metadata.distribution:
        recset_id = obj.get('@id', None)
        if recset_id:
            record_set_ids.append(recset_id)
            print(f"- (Distribution) Record Set/FileObject @id: {recset_id}")

print("\nAll discovered record set IDs:", record_set_ids)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the actual record set and field `@id`s from the overview above.

If you found the RecordSet(s) above, use their `@id`. We'll try to load all possible discovered record sets.

In [ ]:
# Extract data from each record set
dataframes = {}
print("Loading available RecordSets into DataFrames...\n")
# Use the discovered @id's
for record_set_id in record_set_ids:
    try:
        print(f"Loading records for RecordSet: {record_set_id}")
        # By Croissant, the @id can be used as an argument, e.g., dataset.records(record_set=...)
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"- Columns: {df.columns.tolist()}")
            print(f"- Sample data (head):\n{df.head()}\n")
        else:
            print("- No records found for this record set.\n")
    except Exception as e:
        print(f"- Failed to load records from {record_set_id}: {e}\n")

# For further analysis, select one main record set to focus on
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Selected main record set for subsequent analysis: {main_record_set_id}")
    print("Available columns:", dataframes[main_record_set_id].columns.tolist())
else:
    raise ValueError("No record sets could be loaded with data.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records by a numeric criterion, normalizing fields, and grouping by categorical attributes.

We'll demonstrate filtering, normalization, and grouping using a representative numeric and a categorical column. All field/column references are by their `@id`.

In [ ]:
# Identify a numeric and a categorical field by @id for EDA

df = dataframes[main_record_set_id]

# Try to autodetect numeric and group columns
numeric_field_id = None
group_field_id = None
# List available columns
print("Columns in main DataFrame:", df.columns.tolist())
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
# Heuristic for group-by, look for 'Category', 'Type', or similar
for col in df.columns:
    if any(word in col.lower() for word in ['class', 'type', 'category', 'group']) and col != numeric_field_id:
        group_field_id = col
        break
if numeric_field_id is None:
    # Fallback: try forced conversion and guess
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            numeric_field_id = col
            break
        except Exception:
            continue

print(f"Selected numeric field (@id): {numeric_field_id}")
if group_field_id:
    print(f"Selected group field (@id): {group_field_id}")
else:
    print("No explicit group field found.")

# EDA steps
if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id} (means):")
        print(grouped_df.head())
else:
    print("No numeric field detected for EDA analysis.")

## 5. Visualization
Visualize data distributions or relationships between numeric fields in the dataset.

We'll show a histogram of the detected numeric field and, if possible, a grouped barplot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns and df[group_field_id].nunique() < 20:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we explored and analyzed the FAIR^2 mass spectrometry dataset using the mlcroissant library. We:
- Loaded the dataset's Croissant schema and metadata,
- Inspected record sets and field `@id`s for principled referencing,
- Loaded data into Pandas DataFrames,
- Performed initial filtering, normalization, and grouping of records (referencing all fields by Croissant `@id`),
- Visualized distributions of numeric fields.

You can adapt this notebook by referencing different `@id`s for record sets and fields as needed. For advanced analysis, refer to the field documentation and Croissant schema for rich semantics and provenance.